# 内存结构

Polars 的 DataFrame 背后是 Apache Arrow 格式的数组。因此内存效率高，数据访问和操作也快。Apache Arrow 是专为数据分析设计的列式内存格式，特点如下。

1. 内存效率：Apache Arrow 把数据存放在连续的内存块里，缓存友好。处理大量数据时，内存占用也能压到最低。

2. 兼容性：Arrow 格式与许多数据分析工具和库兼容，数据在不同工具之间搬动很顺畅。

3. 快速访问：列式存储让特定列的访问和操作非常快。数据量大、又反复操作某几列时尤其明显。

4. 数据类型丰富：Arrow 支持 NULL、列表、结构体、字符串等多种类型，复杂的数据结构也能直接表达和操作。

本章详述 NULL、字符串、Array、List、Struct 这些数据类型的内存结构。

In [1]:
import struct
import polars as pl

## 数据的管理

Polars 最底层的数据管理用的是 `pyarrow` 的 `Buffer`。下面的代码把 `df['A']` 取两次，用 `id()` 函数比较这两个 `Series` 对象是否相同。

In [2]:
df = pl.DataFrame({
    "A": [1, 2, 3, 4],
    "B": [5, 6, 7, 8]
})

# 取两次 Series 对象
s1 = df['A']
s2 = df['A']
print(id(s1), id(s2))

2396143245648 2396112491344


结果是取到的两个 `Series` 对象**并不相同**。Polars 的 `Series` 内部用的是 Rust 实现的 `PySeries`，而这两个 `PySeries` 也是不同的对象。

In [3]:
print(id(s1._s), id(s2._s))

2396105317456 2396140436768


把每个 `Series` 对象转成 PyArrow 数组，拿到的数组也**并不相同**。

In [4]:
a1 = s1.to_arrow()
a2 = s2.to_arrow()
print(id(a1), id(a2))

2396114046880 2396144045024


再取 PyArrow 数组内部使用的 `Buffer` 对象，ID 不同，说明 `Buffer` 对象本身也不同。但用 `address` 属性看数据地址，会发现**地址是同一个**。下图是各对象之间的关系。

```{mermaid}
flowchart LR
    s1(Series s1) -->|_s| s1s(PySeries)
    s2(Series s2) -->|_s| s2s(PySeries)
    s1s -->|"to_arrow(False)"| s1sa(pyarrow.Array)
    s2s -->|"to_arrow(False)"| s2sa(pyarrow.Array)
    s1 -->|"to_arrow()"| s1sa
    s2 -->|"to_arrow()"| s2sa
    s1sa -->|"buffers()[1]"| s1sab(pyarrow.Buffer)
    s2sa -->|"buffers()[1]"| s2sab(pyarrow.Buffer)
    s1sab -->|address| memory(Data Memory)
    s2sab -->|address| memory(Data Memory)
```

In [5]:
b1 = a1.buffers()[1]
b2 = a2.buffers()[1]
print(id(b1), id(b2))
print(b1.address, b2.address)

2396151175408 2396151175472
2394447088640 2394447088640


接着用索引操作改动 `Series` 对象的元素，再确认 `Buffer` 对象的数据地址。

In [6]:
s1[1] = 100
print(s1.to_arrow().buffers()[1].address)

2394447088000


结果数据的地址**变了**。也就是说，这个操作不是在同一地址上覆写，而是把数据存到新地址。因此 `s2` 和 `df` 还持有原来的数据，看不到 `s1` 的改动。

## NULL 的存储方式

Polars 用 PyArrow 的数据类型保存数据。含 NULL 值的数组用两个 buffer（缓冲区）来存，既能高效管理 NULL，又能保证实际数据完好。

* **mask buffer**（掩码缓冲）：逐位记录各数据元素是否为 NULL。每一位对应一个元素，位为 1 表示非 NULL，为 0 表示 NULL。这种做法省内存，数据量再大，NULL 信息也占不了多少。

* **data buffer**（数据缓冲）：存放实际数据。NULL 的位置一般用 0 或其它占位值填上，实际的数据元素连续存放。

下面看具体例子。

In [7]:
s = pl.Series([1, None, None, 3, 4, None, 8, 9], dtype=pl.Int16)
a = s.to_arrow()
buf_mask, buf_data = a.buffers()

`buf_mask` 用位串表示 NULL 标记，`0b11011001` 的每一位对应一个数据元素的 NULL 状态。从最右一位（第 1 个元素）起依次是 `1, 0, 0, 1, 1, 0, 1, 1`，可见第 2、3、6 个元素是 NULL。

In [8]:
bin(bytes(buf_mask)[0])

'0b11011001'

In [9]:
struct.unpack('<8h', buf_data)

(1, 0, 0, 3, 4, 0, 8, 9)

下面的代码确认 `buf_data` 和 `s` 共享同一块内存。这里用 `ctypes` 把 `buf_data.address` 转成指针 `pbuf`，通过 `pbuf` 把第 4 个值改成 99，最后打印 `s` 看改动是否生效。

In [10]:
from ctypes import POINTER, c_int16, cast
pbuf = cast(buf_data.address, POINTER(c_int16))
pbuf[3] = 99
print(s)

shape: (8,)
Series: '' [i16]
[
	1
	null
	null
	99
	4
	null
	8
	9
]


## String

Polars 的字符串列用三个 buffer 存数据，把内存效率榨到最高。这样字符串的访问和操作都很快。三个 buffer 分别是：

**mask buffer**（掩码缓冲）：逐位记录各字符串元素是否为 NULL。位为 1 表示非 NULL，为 0 表示 NULL，NULL 的存在与否一目了然。

**index buffer**（索引缓冲）：保存每个字符串元素的起始索引和结束索引，可以从连续的字符串数据里快速切出各元素。第 i 个字符串的起始索引是 `buf_index[i]`，结束索引是 `buf_index[i+1]`。

**data buffer**（数据缓冲）：所有字符串连续存放在这里。连续存放把内存占用压到最小，访问也更快。

下面看具体例子。

In [11]:
s = pl.Series(["abc", "defghi", "xyz", None, "123"])
a = s.to_arrow()
buf_mask, buf_index, buf_data = a.buffers()

`buf_mask` 的每一位表示对应字符串元素的 NULL 状态。前三个元素不是 NULL，第 4 个元素是 NULL。

In [12]:
bin(bytes(buf_mask)[0])

'0b11110111'

`buf_index` 里存的是 `[0, 3, 9, 12, 12, 15]`，也就是各字符串元素的起止索引。

* "abc" 的起始索引是 0，结束索引是 3
* "defghi" 的起始索引是 3，结束索引是 9
* 该元素是 NULL，起止索引都是 12

In [13]:
struct.unpack('<6q', buf_index)

(0, 3, 9, 12, 12, 15)

`buf_data` 里所有字符串数据连续存放。借索引缓冲，就能从这段连续数据里快速切出各字符串元素。

In [14]:
buf_data.to_pybytes()

b'abcdefghixyz123'

比如 `s[2]` 的起始索引是 9、结束索引是 12，从 `buf_data` 里取出 `s[2]` 的数据：

In [15]:
buf_data[9:12].to_pybytes()

b'xyz'

## Array

Polars 的 Array 列存固定长度的列表，内存结构和其他列有共通之处，也有自己的东西，`list_size` 属性尤其关键。下面用代码例子说明 Array 列的内存结构。

In [16]:
s = pl.Series([
    [1,   None, 3   ], 
    [4,   5,    None], 
    [6,   7,    8   ], 
    [9,   10,   11  ]], 
    dtype=pl.Array(pl.Int16, 3))
a = s.to_arrow()

`list_size` 属性表示每个元素的长度。

In [17]:
print(f"{a.type.list_size=}")

a.type.list_size=3


`buf_mask` 和 `buf_data` 分别存 NULL 和数据，这两个 buffer 和一般列一样。

In [18]:
_, buf_mask, buf_data = a.buffers()
bin(bytes(buf_mask)[0])

'0b11011101'

In [19]:
struct.unpack('<12h', buf_data)

(1, 0, 3, 4, 5, 0, 6, 7, 8, 9, 10, 11)

(list_memory)=
## List

Polars 的 List 列存变长列表，内存结构是为高效管理数据设计的，用到几个专门的 buffer。这个结构和字符串列（str 列）有相似之处。下面结合代码例子和输出说明。

In [20]:
s = pl.Series([
    [1, None, 3], 
    [10, 20], 
    None, 
    [100, 200, 300]], 
    dtype=pl.List(pl.Int16))
a = s.to_arrow()
buf_mask1, buf_index, buf_mask2, buf_data = a.buffers()

`buf_mask1` 逐位记录每个列表元素是否为 NULL。位为 1 表示非 NULL，为 0 表示 NULL。第 3 位是 0，说明第 3 个元素是 NULL。

In [21]:
bin(bytes(buf_mask1)[0])

'0b11111011'

`buf_index` 记录每个列表元素的起始索引和结束索引。和字符串列（str 列）一样，用它从连续数据里切出各列表元素。

`s[0]` 的数据索引是 0~3，也就是 `buf_data` 里的 `[1, 0, 3]`。`s[1]` 的数据索引是 3~5，即 `buf_data` 里的 `[10, 20]`。

In [22]:
struct.unpack('<5Q', buf_index)

(0, 3, 5, 5, 8)

下一个 buffer 逐位记录列表内每个元素是否为 NULL。第 2 位是 0，说明第 2 个元素（None）是 NULL。

In [23]:
bin(bytes(buf_mask2)[0])

'0b11111101'

所有列表元素的实际数据连续存在 `buf_data` 里，NULL 值用占位值（本例中是 0）表示。

In [24]:
struct.unpack('<8h', buf_data)

(1, 0, 3, 10, 20, 100, 200, 300)

## Struct

Struct 列存带多个字段的数据。每个字段有自己的数据类型和内存结构，用几个 buffer 来高效管理。下面用代码例子说明 Struct 列的内存结构。

In [25]:
s = pl.Series([{"A": 1, "B": None}, {"A": None, "B": 20}, {"A": 3, "B": 30}, None])
a = s.to_arrow()
a.buffers()

[<pyarrow.Buffer address=0x22d80210a50 size=1 is_cpu=True is_mutable=False>,
 <pyarrow.Buffer address=0x22d80210a48 size=1 is_cpu=True is_mutable=False>,
 <pyarrow.Buffer address=0x22d802b04e0 size=32 is_cpu=True is_mutable=False>,
 <pyarrow.Buffer address=0x22d80210a40 size=1 is_cpu=True is_mutable=False>,
 <pyarrow.Buffer address=0x22d802b04a0 size=32 is_cpu=True is_mutable=False>]

一个字段用两个 buffer：一个存掩码，一个存值。从这种内存结构可以看出，各字段的值在内存里是连续存放的。

In [26]:
buf_mask_A, buf_value_A = a.buffers()[1:3]
print(bin(bytes(buf_mask_A)[0]))
print(struct.unpack('<4Q', buf_value_A))

0b101
(1, 0, 3, 0)


In [27]:
buf_mask_B, buf_value_B = a.buffers()[3:5]
print(bin(bytes(buf_mask_B)[0]))
print(struct.unpack('<4Q', buf_value_B))

0b110
(0, 20, 30, 0)
